# 01 — Oracle census

Provenance first: the cell below reads this loop's `manifest.json` and prints it.
If it raises, record provenance before computing anything.

In [1]:
from pathlib import Path

import pandas as pd
from metron import deviation_vs_reference, log_returns, realized_vol, staleness_stats

from mrsearch import SnapshotReader, read_manifest

manifest = read_manifest(Path.cwd())
print(manifest)

r = SnapshotReader(Path.cwd().parents[1] / "data")
pd.set_option("display.width", 200)

Manifest(snapshot_date='2026-08-07', mnemon_commit='abde3c1d60b2ba591aeec0978ccc60cd846b5c39', metron_version='v1.0.0', tables=('market_state', 'markets', 'prices', 'market_flows'))


## Lane 1 — fingerprint census

Per market, from live 5-min `oracle_price` samples: METRON `staleness_stats`
(stale = exact equality with the previous sample), `realized_vol` on log
returns, and the share of upward moves among nonzero changes
(hand-tabulated — `TODO(metron): add direction_stats`). The class column is a
descriptive heuristic, not a prediction: `frozen` (no changes at all),
`exchange-rate` (accrual: >=90% of changes upward), `nav/sparse` (>=95%
stale but two-sided), `market-rate` (everything else).

In [2]:
df = r.sql("""
    SELECT ts, market_id, loan_symbol, collateral_symbol, oracle_price
    FROM v_market_state
    WHERE oracle_price IS NOT NULL AND collateral_symbol IS NOT NULL
    ORDER BY market_id, ts
""")


def classify(stale_share, n_changes, up_share):
    if n_changes == 0:
        return "frozen"
    if up_share == up_share and up_share >= 0.9:
        return "exchange-rate"
    if stale_share >= 0.95:
        return "nav/sparse"
    return "market-rate"


rows = []
for mid, g in df.groupby("market_id"):
    s = g.drop_duplicates(subset="ts").set_index("ts")["oracle_price"].astype(float)
    s.index = pd.DatetimeIndex(s.index)
    if len(s) < 100 or (s <= 0).any():
        continue
    st = staleness_stats(s)
    lr = log_returns(s)
    nz = lr[lr != 0]
    up = float((nz > 0).mean()) if len(nz) > 10 else float("nan")
    n_changes = int(s.ne(s.shift()).iloc[1:].sum())
    rows.append(
        {
            "market": f"{g['collateral_symbol'].iloc[0]}/{g['loan_symbol'].iloc[0]}",
            "market_id": mid[:10],
            "n_obs": st.n_obs,
            "stale_share": round(st.stale_share, 3),
            "n_changes": n_changes,
            "longest_stale_h": round(st.longest_stale.total_seconds() / 3600, 1),
            "vol_per_5m": realized_vol(lr),
            "up_share": round(up, 3),
            "class": classify(st.stale_share, n_changes, up),
        }
    )

census = pd.DataFrame(rows).sort_values("stale_share", ascending=False)
print(census.to_string(index=False))
print()
print(census["class"].value_counts().to_string())

                   market  market_id  n_obs  stale_share  n_changes  longest_stale_h   vol_per_5m  up_share         class
               AVLT/USD₮0 0x8eecdd03   4906        1.000          0            408.8 0.000000e+00       NaN        frozen
             hbHYPE/WHYPE 0x19e47d37   7014        0.999          0            695.8 0.000000e+00       NaN        frozen
 PT-kHYPE-13NOV2025/WHYPE 0x1df0d0eb   7014        0.999          0            695.8 0.000000e+00       NaN        frozen
             beHYPE/WHYPE 0xae019cf2   4906        0.999          2            307.9 1.894701e-06       NaN    nav/sparse
              hbUSDT/USDC 0x70c171a5   4906        0.998          8            122.9 1.392233e-05       NaN    nav/sparse
              hbUSDT/USDe 0x7268244d   4906        0.998          8            122.9 1.392233e-05       NaN    nav/sparse
             hbUSDT/USD₮0 0x2acd218c   7014        0.997         11            122.9 1.612504e-05     1.000 exchange-rate
             hbUSDT/USD₮

## Lane 2 — divergence screen

MNEMON's `v_oracle_price_check` provides the per-row algebra (oracle vs
ASOF-joined DeFiLlama reference, both as "1 collateral in loan tokens");
METRON's `deviation_vs_reference` provides the statistics (rule-5
time-weighted mean, max with timestamp). `v_depeg_spells` identifies
episodes of |deviation| >= 2% / 5% with 2h hole tolerance. Deviation can
come from either leg: a loan-token depeg shows as persistent negative
deviation (see sUSDe/USH).

In [3]:
chk = r.sql("""
    SELECT ts, market_id, loan_symbol, collateral_symbol, oracle_price, ref_price
    FROM v_oracle_price_check
    WHERE ref_price IS NOT NULL AND oracle_price IS NOT NULL AND ref_price > 0
    ORDER BY market_id, ts
""")

rows = []
for mid, g in chk.groupby("market_id"):
    g = g.drop_duplicates(subset="ts").set_index("ts")
    if len(g) < 100:
        continue
    idx = pd.DatetimeIndex(g.index)
    d = deviation_vs_reference(
        pd.Series(g["oracle_price"].astype(float).to_numpy(), index=idx),
        pd.Series(g["ref_price"].astype(float).to_numpy(), index=idx),
    )
    rows.append(
        {
            "market": f"{g['collateral_symbol'].iloc[0]}/{g['loan_symbol'].iloc[0]}",
            "market_id": mid[:10],
            "n_obs": d.n_obs,
            "mean_dev_pct": round(100 * d.mean_deviation, 2),
            "max_abs_dev_pct": round(100 * d.max_abs_deviation, 2),
            "max_dev_time": d.max_abs_deviation_time,
        }
    )

divergence = pd.DataFrame(rows).sort_values("max_abs_dev_pct", ascending=False)
print(divergence.head(20).to_string(index=False))

spells = r.sql("""
    SELECT m.collateral_symbol || '/' || m.loan_symbol AS market, s.threshold,
           COUNT(*) AS n_spells, MAX(s.duration_min) AS max_dur_min,
           ROUND(100 * MAX(s.peak_abs_deviation), 1) AS peak_pct
    FROM v_depeg_spells s JOIN markets m USING (chain_id, market_id)
    GROUP BY 1, 2 HAVING MAX(s.duration_min) >= 60 ORDER BY peak_pct DESC
""")
print()
print(spells.to_string(index=False))

                  market  market_id  n_obs  mean_dev_pct  max_abs_dev_pct              max_dev_time
              AVLT/USD₮0 0x8eecdd03   4906        195.55          1034.97 2026-08-02 19:15:00+00:00
               sUSDe/USH 0xb9654602   2308        -17.56            31.04 2026-07-21 15:25:00+00:00
              kHYPE/USDH 0x71374f58   4906         -0.68             7.63 2026-07-22 07:20:00+00:00
               UBTC/USDH 0x1f97b631   3340         -0.74             7.59 2026-07-22 07:20:00+00:00
              WHYPE/USDH 0x85e7ea4f   4906         -0.72             7.54 2026-07-22 07:20:00+00:00
            wstHYPE/USDH 0xa6ddbd0e    725         -3.63             4.66 2026-07-23 19:55:00+00:00
 PT-kHYPE-24SEP2026/USDC 0xbdceb936   4906         -1.18             4.40 2026-07-29 19:35:00+00:00
PT-kHYPE-24SEP2026/USD₮0 0x13843ab7   4906         -1.17             4.40 2026-07-29 19:35:00+00:00
            wstHYPE/USDC 0xcac72237   4906         -2.93             4.15 2026-08-02 10:25:00+00:00



       market  threshold  n_spells  max_dur_min  peak_pct
   AVLT/USD₮0       0.02         1        24525    1035.0
   AVLT/USD₮0       0.05        12        16735    1035.0
    sUSDe/USH       0.05         1        11550      31.0
    sUSDe/USH       0.02         1        11550      31.0
    UBTC/USDH       0.05         2           80       7.6
    UBTC/USDH       0.02        15          535       7.6
   kHYPE/USDH       0.02        31          410       7.6
   WHYPE/USDH       0.02        30          565       7.5
 wstHYPE/USDH       0.02         1         3700       4.7
 wstHYPE/USDC       0.02         9        15545       4.1
lstHYPE/WHYPE       0.02        15          310       3.6
 hbHYPE/WHYPE       0.02        13          225       3.2
 thBILL/USD₮0       0.02        12          255       2.3


## Lane 3 — calibration anatomy: AVLT/USDT0 through the 2026-06-21 depeg

Reference price from `prices` (source `morpho_history` then `llama_fine`;
DeFiLlama listed AVLT 2026-03-25), flows from `market_flows`. The oracle
(issuer NAV) has no pre-2026-07-21 samples; it enters as the frozen
post-depeg assertion. PT-decay markets serve as the stale-looking-but-benign
control (see census: PT-kHYPE-24SEP2026 updates every sample, ~monotone).

In [4]:
AVLT_MKT = "0x8eecdd03f1e12e03c04abefdb3f536067e071ad2c3f63e7b04c9a034889d0ba5"
AVLT_TOK = "0xd0ee0cf300dfb598270cd7f4d0c6e0d8f6e13f29"

print("--- reference price, weekly medians")
print(
    r.sql(
        """
    SELECT DATE_TRUNC('week', ts) AS week, ROUND(MEDIAN(price_usd), 4) AS median_px
    FROM prices WHERE token_address = ? AND ts >= '2026-06-01'
    GROUP BY 1 ORDER BY 1
""",
        [AVLT_TOK],
    ).to_string(index=False)
)

print()
print("--- weekly loan-side flows (USDT0)")
print(
    r.sql(
        """
    SELECT DATE_TRUNC('week', ts) AS week, type,
           ROUND(SUM(assets::DOUBLE) / 1e6) AS usdt0,
           COUNT(DISTINCT account) AS accounts
    FROM market_flows
    WHERE market_id = ? AND type IN ('Supply', 'Withdraw', 'Borrow', 'Repay')
      AND ts >= '2026-06-01'
    GROUP BY 1, 2 ORDER BY 1, 2
""",
        [AVLT_MKT],
    ).to_string(index=False)
)

print()
print("--- market state, weekly")
print(
    r.sql(
        """
    SELECT DATE_TRUNC('week', ts) AS week,
           ROUND(AVG(supply_assets)) AS supply, ROUND(AVG(borrow_assets)) AS borrow,
           ROUND(100 * AVG(utilization), 1) AS util_pct
    FROM v_market_state WHERE market_id = ? AND ts >= '2026-06-01'
    GROUP BY 1 ORDER BY 1
""",
        [AVLT_MKT],
    ).to_string(index=False)
)

print()
print("--- liquidations DID fire: weekly, with the implied seize price")
print(
    r.sql(
        """
    SELECT DATE_TRUNC('week', ts) AS week, COUNT(*) AS n_liqs,
           COUNT(DISTINCT liquidator) AS liquidators,
           ROUND(SUM(repaid_assets::DOUBLE) / 1e6, 1) AS repaid_usdt0,
           ROUND(SUM(repaid_assets::DOUBLE) / NULLIF(SUM(seized_assets::DOUBLE), 0), 5)
               AS implied_seize_px,
           SUM(bad_debt_assets::DOUBLE) / 1e6 AS bad_debt
    FROM market_flows WHERE market_id = ? AND type = 'Liquidation'
    GROUP BY 1 ORDER BY 1
""",
        [AVLT_MKT],
    ).to_string(index=False)
)

# Morpho's liquidation incentive factor for this market's LLTV. The seize
# price above should equal oracle / LIF exactly if liquidations executed at
# the frozen oracle mark.
LLTV, ORACLE = 0.915, 1.094493
lif = 1 / (1 - 0.3 * (1 - LLTV))
print(f"\nLIF({LLTV}) = {lif:.5f}; oracle / LIF = {ORACLE / lif:.5f}")

print()
print("--- borrower health factors at the frozen oracle mark (weekly)")
print(
    r.sql(
        """
    SELECT DATE_TRUNC('week', ts) AS week, COUNT(DISTINCT borrower) AS borrowers,
           ROUND(MEDIAN(health_factor), 3) AS median_hf
    FROM positions WHERE market_id = ? AND health_factor IS NOT NULL
    GROUP BY 1 ORDER BY 1
""",
        [AVLT_MKT],
    ).to_string(index=False)
)

print()
print("--- recorded bad debt, whole chain, all time")
print(
    r.sql("""
    SELECT COUNT(*) AS events,
           ROUND(SUM(f.bad_debt_assets::DOUBLE / POW(10, m.loan_decimals)), 2) AS loan_units
    FROM market_flows f JOIN markets m USING (chain_id, market_id)
    WHERE f.bad_debt_assets > 0
""").to_string(index=False)
)

--- reference price, weekly medians
                     week  median_px
2026-06-01 00:00:00+00:00     1.0833
2026-06-08 00:00:00+00:00     1.0875
2026-06-15 00:00:00+00:00     1.0902
2026-06-22 00:00:00+00:00     0.7967
2026-06-29 00:00:00+00:00     0.6897
2026-07-06 00:00:00+00:00     0.8393
2026-07-13 00:00:00+00:00     0.8872
2026-07-20 00:00:00+00:00     1.0256
2026-07-27 00:00:00+00:00     0.3067
2026-08-03 00:00:00+00:00     0.2457

--- weekly loan-side flows (USDT0)
                     week     type      usdt0  accounts
2026-06-01 00:00:00+00:00   Borrow  2590070.0        72
2026-06-01 00:00:00+00:00    Repay  2971654.0        47
2026-06-01 00:00:00+00:00   Supply  6126858.0         4
2026-06-01 00:00:00+00:00 Withdraw  6150100.0         4
2026-06-08 00:00:00+00:00   Borrow  3954017.0        55
2026-06-08 00:00:00+00:00    Repay  1460994.0        33
2026-06-08 00:00:00+00:00   Supply  6189141.0         3
2026-06-08 00:00:00+00:00 Withdraw  3957283.0         3
2026-06-15 00:00:

## Reading

- **Lane 1** separates four behaviors on one month of data: frozen oracles
  (AVLT, hbHYPE/WHYPE, matured PTs), exchange-rate accrual (LSTs, hbUSDT),
  sparse-but-two-sided heartbeat oracles (UBTC/UETH/WHYPE families), and
  continuously-updating composites (live PTs).
- **Lane 2** flags AVLT (mean +196%, max +1035% — the assertion gap), the
  USH loan-token depeg (-31%, 8-day spell), a kHYPE ~4% market dip on
  2026-07-29 that every kHYPE exchange-rate oracle asserted par through,
  and structural premia (thBILL) that are features, not alerts.
- **Lane 3**: borrowing against NAV-marked AVLT was the rational exit —
  borrows accelerated into the depeg; ~7.5M of 9.8M USDT0 escaped in a
  four-week rotation; 2.2M remains at 100% utilization against collateral
  worth ~10-25% of its oracle mark.
- **Liquidations are not blocked by the frozen oracle.** 750 fired from
  2026-06-25 on, at exactly the oracle mark: the implied seize price equals
  oracle / LIF to five decimals. Interest accrual (639% APY at target,
  100% utilization) — not collateral repricing — is what pushes positions
  past LLTV, and median health factor keeps falling (0.78 -> 0.60 over
  three weeks) while 253 borrowers sit liquidatable-but-unliquidated,
  because repaying real USDT0 for AVLT marked 4x over market is a
  guaranteed loss to the liquidator.
- **Recorded bad debt is nonetheless zero here**, because Morpho books bad
  debt only when a liquidation exhausts a position's collateral and debt
  remains. At an inflated mark the collateral always *appears* sufficient,
  so the shortfall is never booked — it is transferred to whoever chose to
  liquidate. Realization would require the oracle converging to market, or
  interest accruing past what even the frozen mark can cover.